<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/Rockets_Escape_Velocity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project: Escape Velocity Visualization

## Overview
This notebook contains a high-fidelity physics simulation and animation created using the Manim library. The project visualizes the concept of escape velocity through eight sequential launch attempts, demonstrating how different initial speeds affect a craft's trajectory relative to a planetary body.

## Physics Background

### Escape Velocity
Escape velocity is the minimum speed an object must reach to break free from the gravitational attraction of a massive body without further propulsion. For a spherical body with mass $M$ and radius $r$, the escape velocity $v_{esc}$ is calculated as:

$$v_{esc} = \sqrt{\frac{2GM}{r}}$$

where $G$ is the gravitational constant.

### Orbital Mechanics and Energy States
The behavior of the rocket in this simulation is determined by its specific orbital energy ($E$), which is the sum of its kinetic and potential energy:

1. **Suborbital (Bound, $E < 0$):** When $v < v_{orb}$, the craft returns to the surface.
2. **Circular Orbit (Bound, $E < 0$):** At $v = v_{esc} / \sqrt{2}$, the craft maintains a stable circular path.
3. **Elliptical Orbit (Bound, $E < 0$):** Between circular and escape velocity, the craft follows a closed elliptical loop.
4. **Parabolic Trajectory (Critical, $E = 0$):** At exactly $v = v_{esc}$, the craft is on the threshold of escaping.
5. **Hyperbolic Trajectory (Unbound, $E > 0$):** At $v > v_{esc}$, the craft escapes the gravity well and follows an open path into deep space.

## Technical Implementation

### Tools Used
- **Manim (Mathematical Animation Engine):** Used for rendering the 9:16 vertical video optimized for mobile viewing.
- **NumPy:** Utilized for trajectory simulation and vector mathematics.
- **IPython Display:** Used to embed the rendered MP4 output directly within the notebook environment.

### Simulation Details
The simulation accounts for a curved ascent phase (injection) followed by gravitational propagation. The visual layout uses a parchment-style aesthetic with specialized cards to display real-time statistics and mathematical formulas for each launch attempt.

In [1]:
!apt-get update -qq
!apt-get install -y -qq libcairo2-dev libpango1.0-dev ffmpeg dvisvgm texlive-latex-extra texlive-fonts-extra
!pip install -q manim


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Preconfiguring packages ...
Selecting previously unselected package fonts-droid-fallback.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../000-fonts-droid-fallback_1%3a6.0.1r16-1.1build1_all.deb ...
Unpacking fonts-droid-fallback (1:6.0.1r16-1.1build1) ...
Selecting previously unselected package fonts-lato.
Preparing to unpack .../001-fonts-lato_2.0-2.1_all.deb ...
Unpacking fonts-lato (2.0-2.1) ...
Selecting previously unselected package poppler-data.
Preparing to unpack .../002-poppler-data_0.4.11-1_all.deb ...
Unpacking poppler-data (0.4.11-1) ...
Selecting previously unselected package tex-common.
Preparing to unpack .../003-tex-common_6.17_all.deb ...
Unpacking tex-common (6.17) ...
Selecting previously unse

In [1]:
import manim
from manim.utils.ipython_magic import ManimMagic

try:
    # Manually register the %%manim magic command into the IPython shell
    get_ipython().register_magics(ManimMagic)
    print(f"Manim {manim.__version__} loaded and magic commands registered successfully.")
except Exception as e:
    print(f"Error loading Manim magic: {e}")

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Manim 0.20.1 loaded and magic commands registered successfully.


In [5]:
%%manim -r 540,960 EscapeVelocityStory
"""
Author: Mugambi Ndwiga
Instagram: @craftsandengineering
Concept: Escape Velocity — Integrated Layout with Trajectory Labeling
"""

from manim import *
import numpy as np
import random

# Core system configuration for 9:16 vertical viewports
config.pixel_width = 540
config.pixel_height = 960
config.frame_width = 6.0
config.frame_height = config.frame_width * (960 / 540)
config.frame_rate = 24
config.background_color = "#efe2c3"

PARCHMENT = "#efe2c3"
PARCHMENT_LIGHT = "#f6ebd4"
PARCHMENT_DARK = "#e6d4ae"
INK = "#4b3124"
INK_FAINT = "#7c5e49"
ACCENT = "#8b4b3b"
SUCCESS = ManimColor("#5e6e4c")
FAIL = ManimColor("#8b5a4b")
GOLD = "#a87d45"

# Dual-Font System: Display elegance combined with flawless UI legibility
TITLE_FONT = "TeX Gyre Chorus"
BODY_FONT = "Liberation Sans"

N_ATTEMPTS = 8
launch_speed_factors = [0.55, 0.62, 0.71, 0.82, 1.00, 1.05, 1.15, 1.25]
attempt_captions = [
    "Sub-orbital: Gravity wins.",
    "Sub-orbital: Higher arc.",
    "Stable circular orbit.",
    "Highly elliptical orbit.",
    "Parabolic escape limit.",
    "Hyperbolic escape path.",
    "Escaping easily.",
    "Beyond the gravity well.",
]

SIM_RADIUS = 0.65
SIM_SCALE = 1.60
MU = 0.5 * SIM_RADIUS

EARTH_CENTER = UP * 1.45 + LEFT * 0.0
EARTH_RADIUS = SIM_RADIUS * SIM_SCALE

LAUNCH_SURFACE_ANGLE = 55 * DEGREES
LAUNCH_POINT_SIM = SIM_RADIUS * np.array([
    np.cos(LAUNCH_SURFACE_ANGLE),
    np.sin(LAUNCH_SURFACE_ANGLE),
    0.0
])
LAUNCH_POINT = EARTH_CENTER + SIM_SCALE * LAUNCH_POINT_SIM

def sim_to_scene(points):
    return [EARTH_CENTER + SIM_SCALE * p for p in points]

def get_classification(factor):
    """Explicit requested velocity regime classification."""
    if factor < 0.71:
        return "Sub-orbital", FAIL
    elif abs(factor - 0.71) < 1e-2 or factor < 1.0:
        return "Orbital", GOLD if abs(factor - 0.71) < 1e-2 else SUCCESS
    else:
        return "Escape Velocity", ACCENT

def escape_status(factor):
    regime, color = get_classification(factor)
    if abs(factor - 1.0) < 1e-9:
        return regime.upper(), color, "E = 0 (Parabolic)"
    if abs(factor - 0.71) < 1e-2:
        return regime.upper(), color, "v = vorb (Circular)"
    if factor < 0.71:
        return regime.upper(), color, "E < 0 (Suborbital)"
    if factor < 1.0:
        return regime.upper(), color, "E < 0 (Elliptic)"
    return regime.upper(), color, "E > 0 (Hyperbolic)"

def simulate_attempt(speed_factor, dt=0.004, max_steps=12000):
    theta_0 = 55 * DEGREES
    theta_1 = 75 * DEGREES
    R_0 = SIM_RADIUS
    R_1 = SIM_RADIUS * 1.35

    P_0 = R_0 * np.array([np.cos(theta_0), np.sin(theta_0), 0.0])
    P_3 = R_1 * np.array([np.cos(theta_1), np.sin(theta_1), 0.0])

    L_1 = 0.12
    L_2 = 0.18

    P_1 = P_0 + L_1 * np.array([np.cos(theta_0), np.sin(theta_0), 0.0])
    P_2 = P_3 - L_2 * np.array([-np.sin(theta_1), np.cos(theta_1), 0.0])

    ascent_points = []
    num_ascent_steps = 45
    for t in np.linspace(0, 1, num_ascent_steps):
        B_t = (1-t)**3 * P_0 + 3*(1-t)**2 * t * P_1 + 3*(1-t) * t**2 * P_2 + t**3 * P_3
        ascent_points.append(B_t)

    p = P_3.copy()
    r_inj = np.linalg.norm(p)
    v_esc_local = np.sqrt(2 * MU / r_inj)
    v_mag = speed_factor * v_esc_local
    v = v_mag * np.array([-np.sin(theta_1), np.cos(theta_1), 0.0])

    points = list(ascent_points)
    initial_angle = np.arctan2(p[1], p[0])
    total_angle_changed = 0.0
    last_angle = initial_angle

    for i in range(max_steps):
        r = np.linalg.norm(p)
        a = -MU * p / (r ** 3)
        v += a * dt
        p += v * dt
        points.append(p.copy())

        current_angle = np.arctan2(p[1], p[0])
        diff = current_angle - last_angle
        if diff > PI: diff -= 2*PI
        if diff < -PI: diff += 2*PI
        total_angle_changed += diff
        last_angle = current_angle

        if speed_factor < 0.70 and r <= SIM_RADIUS:
            break
        if speed_factor in [0.71, 0.82] and abs(total_angle_changed) >= 2 * PI:
            break
        if speed_factor >= 1.0 and r > SIM_RADIUS * 4.8:
            break

    return np.array(points)

def make_paper_texture(scene, seed=9):
    rng = random.Random(seed)
    for _ in range(90):
        x = rng.uniform(-config.frame_width / 2, config.frame_width / 2)
        y = rng.uniform(-config.frame_height / 2, config.frame_height / 2)
        d = Dot(point=np.array([x, y, 0.0]), radius=rng.uniform(0.006, 0.014))
        d.set_fill(color=INK_FAINT, opacity=rng.uniform(0.03, 0.07))
        d.set_stroke(width=0)
        scene.add(d)

def make_card(width, height, stroke_color=INK, fill_color=PARCHMENT_LIGHT, fill_opacity=0.92):
    shadow = RoundedRectangle(width=width, height=height, corner_radius=0.10)
    shadow.set_fill(BLACK, opacity=0.05).set_stroke(width=0).shift(0.03 * RIGHT + 0.03 * DOWN)
    body = RoundedRectangle(width=width, height=height, corner_radius=0.10)
    body.set_fill(fill_color, opacity=fill_opacity).set_stroke(stroke_color, width=1.2)
    return VGroup(shadow, body)

def make_rocket():
    body = RoundedRectangle(width=0.30, height=0.11, corner_radius=0.03).set_fill(INK, opacity=1.0).set_stroke(width=0)
    nose = Triangle().scale(0.075).rotate(PI / 2).set_fill(INK, opacity=1.0).set_stroke(width=0).next_to(body, RIGHT, buff=-0.02)
    window = Circle(radius=0.018).set_fill(PARCHMENT_LIGHT, opacity=1.0).set_stroke(width=0).move_to(body.get_center() + RIGHT * 0.02)
    fin_top = Polygon(body.get_left() + DOWN * 0.01, body.get_left() + LEFT * 0.05 + UP * 0.05, body.get_left() + RIGHT * 0.02 + UP * 0.025).set_fill(ACCENT, opacity=0.95).set_stroke(width=0)
    fin_bottom = Polygon(body.get_left() + UP * 0.01, body.get_left() + LEFT * 0.05 + DOWN * 0.05, body.get_left() + RIGHT * 0.02 + DOWN * 0.025).set_fill(ACCENT, opacity=0.95).set_stroke(width=0)
    flame = Polygon(body.get_left() + LEFT * 0.05, body.get_left() + LEFT * 0.12 + UP * 0.03, body.get_left() + LEFT * 0.14, body.get_left() + LEFT * 0.12 + DOWN * 0.03).set_fill(GOLD, opacity=0.80).set_stroke(width=0)
    return VGroup(fin_top, fin_bottom, body, window, nose, flame)

def make_faded_path(points_scene, color, dashed=True):
    path = VMobject()
    path.set_points_smoothly(points_scene)
    if dashed:
        path = DashedVMobject(path, num_dashes=max(16, len(points_scene) // 8))
    path.set_stroke(color=color, width=1.4, opacity=0.20)
    return path

def make_active_path(points_scene, color):
    path = VMobject()
    path.set_points_smoothly(points_scene)
    path.set_stroke(color=color, width=3.2, opacity=0.95)
    return path

def path_tangent_angle(path, t):
    t1 = max(0.0, t - 0.012)
    t2 = min(1.0, t + 0.012)
    p1 = path.point_from_proportion(t1)
    p2 = path.point_from_proportion(t2)
    vec = p2 - p1
    if np.linalg.norm(vec) < 1e-6:
        return 0.0
    return np.arctan2(vec[1], vec[0])

def fit_inside_card(vgroup, max_w, max_h):
    scale_factor = 1.0
    if vgroup.width > max_w:
        scale_factor = min(scale_factor, max_w / vgroup.width)
    if vgroup.height > max_h:
        scale_factor = min(scale_factor, max_h / vgroup.height)
    if scale_factor < 1.0:
        vgroup.scale(scale_factor)
    return vgroup

def stats_content(width, height, attempt_index, factor):
    status, status_color, energy_line = escape_status(factor)
    caption = attempt_captions[attempt_index - 1]

    # Corrected weight parameter assignment
    title = Text("This Launch", font=BODY_FONT, font_size=13, color=INK, weight=BOLD)
    attempt = Text(f"Attempt {attempt_index} / {N_ATTEMPTS}", font=BODY_FONT, font_size=10, color=INK_FAINT)
    ratio = Text(f"v0 / vesc = {factor:.2f}", font=BODY_FONT, font_size=12, color=INK)

    # Corrected weight parameter assignment
    status_text = Text(status, font=BODY_FONT, font_size=12, color=status_color, weight=BOLD)
    energy = Text(energy_line, font=BODY_FONT, font_size=10, color=INK_FAINT)
    note = Text(caption, font=BODY_FONT, font_size=10, color=INK_FAINT)

    bar_bg = RoundedRectangle(width=width - 0.40, height=0.06, corner_radius=0.02).set_fill(color=BLACK, opacity=0.05).set_stroke(color=INK_FAINT, width=0.6, opacity=0.5)
    fill_ratio = min(factor / 1.25, 1.0)
    bar_fill = RoundedRectangle(width=max(0.01, (width - 0.40) * fill_ratio), height=0.06, corner_radius=0.02).set_fill(color=status_color, opacity=0.8).set_stroke(width=0)
    bar_fill.align_to(bar_bg, LEFT)

    marker_x = bar_bg.get_left()[0] + (width - 0.40) * (1.0 / 1.25)
    marker = Line(UP * 0.05, DOWN * 0.05).set_stroke(color=INK, width=1.2).move_to([marker_x, bar_bg.get_center()[1], 0])

    threshold_label = Text("1.00 Threshold", font=BODY_FONT, font_size=8, color=INK_FAINT)
    bar_system = VGroup(bar_bg, bar_fill, marker)
    bar_block = VGroup(bar_system, threshold_label).arrange(DOWN, buff=0.03, aligned_edge=LEFT)

    group = VGroup(title, attempt, ratio, status_text, energy, bar_block, note)
    group.arrange(DOWN, buff=0.06, aligned_edge=LEFT)

    return fit_inside_card(group, width - 0.35, height - 0.30)

def formula_content(width, height):
    # Corrected weight parameter assignment
    head = Text("Why It Works", font=BODY_FONT, font_size=13, color=INK, weight=BOLD)
    eq1 = MathTex(r"v_{\mathrm{esc}} = \sqrt{\frac{2GM}{r}}", font_size=14, color=INK)
    eq2 = MathTex(r"E = \frac{1}{2}v^2 - \frac{GM}{r}", font_size=14, color=INK)
    eq1.set_color_by_tex(r"v_{\mathrm{esc}}", ACCENT)
    eq2.set_color_by_tex(r"E", ACCENT)
    note = Text("vorb = vesc / SQRT(2)", font=BODY_FONT, font_size=10, color=INK_FAINT)

    group = VGroup(head, eq1, eq2, note).arrange(DOWN, buff=0.08, aligned_edge=LEFT)

    return fit_inside_card(group, width - 0.35, height - 0.30)

class EscapeVelocityStory(Scene):
    def construct(self):
        make_paper_texture(self, seed=9)

        watermark = Text("© Mugambi Ndwiga / @craftsandengineering", font=BODY_FONT, font_size=9, color=INK_FAINT).set_opacity(0.4)
        watermark.to_edge(DOWN, buff=0.10)
        self.add(watermark)

        title = Text("When does a rocket escape?", font=TITLE_FONT, font_size=28, color=INK)
        subtitle = Text("Eight sequential launches. Two critical limits.", font=BODY_FONT, font_size=13, color=INK_FAINT)
        title_block = VGroup(title, subtitle).arrange(DOWN, buff=0.06).to_edge(UP, buff=0.3)
        self.play(FadeIn(title_block, shift=0.15 * DOWN), run_time=0.8)

        earth_shadow = Circle(radius=EARTH_RADIUS * 1.02).set_fill(BLACK, opacity=0.05).set_stroke(width=0).move_to(EARTH_CENTER + 0.04 * RIGHT + 0.03 * DOWN)
        earth = Circle(radius=EARTH_RADIUS).set_fill(PARCHMENT_DARK, opacity=1.0).set_stroke(color=INK, width=1.5).move_to(EARTH_CENTER)

        launch_site = Dot(point=LAUNCH_POINT, radius=0.025, color=ACCENT)
        launch_rim = Circle(radius=0.05).set_stroke(color=ACCENT, width=0.8, opacity=0.6).move_to(LAUNCH_POINT)

        self.play(FadeIn(earth_shadow), FadeIn(earth), FadeIn(launch_site), FadeIn(launch_rim), run_time=1.0)

        panel_w = 2.70
        panel_h = 2.20
        formula_card = make_card(panel_w, panel_h, fill_color=PARCHMENT_LIGHT).move_to(LEFT * 1.45 + DOWN * 3.75)
        stats_card = make_card(panel_w, panel_h, fill_color=PARCHMENT_LIGHT).move_to(RIGHT * 1.45 + DOWN * 3.75)

        formula = formula_content(panel_w, panel_h).move_to(formula_card[1].get_center())
        self.play(FadeIn(formula_card), FadeIn(formula), run_time=0.7)

        stats = stats_content(panel_w, panel_h, 1, launch_speed_factors[0]).move_to(stats_card[1].get_center())
        self.play(FadeIn(stats_card), FadeIn(stats), run_time=0.7)
        current_stats = stats

        threshold_note = Text("Velocity dictates geometry: closed loops or unbound escapes.", font=BODY_FONT, font_size=11, color=INK)
        threshold_note.scale_to_fit_width(config.frame_width - 0.6).move_to(DOWN * 2.25)
        self.play(FadeIn(threshold_note), run_time=0.5)

        regime_labels_placed = set()

        for i, factor in enumerate(launch_speed_factors, start=1):
            new_stats = stats_content(panel_w, panel_h, i, factor).move_to(stats_card[1].get_center())
            self.play(FadeOut(current_stats, shift=UP * 0.02), FadeIn(new_stats, shift=UP * 0.02), run_time=0.25)
            current_stats = new_stats

            sim_points = simulate_attempt(factor)
            scene_points = sim_to_scene(sim_points)

            regime_name, color_mix = get_classification(factor)

            active_path = make_active_path(scene_points, color_mix)
            alpha = ValueTracker(0.0)

            def oriented_rocket():
                mob = make_rocket()
                t = alpha.get_value()
                mob.rotate(path_tangent_angle(active_path, t))
                mob.move_to(active_path.point_from_proportion(t))
                return mob

            rocket = always_redraw(oriented_rocket)

            if factor < 0.70:
                run_time = 3.5
            elif factor in [0.71, 0.82]:
                run_time = 5.5
            else:
                run_time = 4.5

            self.add(active_path, rocket)
            self.play(
                Create(active_path),
                alpha.animate.set_value(1.0),
                run_time=run_time,
                rate_func=smooth
            )

            if regime_name not in regime_labels_placed:
                label_pos = scene_points[int(len(scene_points) * 0.65)]

                if regime_name == "Sub-orbital":
                    label_pos = scene_points[np.argmax([p[1] for p in scene_points])] + UP * 0.18 + LEFT * 0.1
                elif regime_name == "Orbital":
                    label_pos = EARTH_CENTER + UP * (EARTH_RADIUS + 0.35) + LEFT * 1.1
                elif regime_name == "Escape Velocity":
                    label_pos = scene_points[-15] + RIGHT * 0.45 + DOWN * 0.2

                # Corrected weight parameter assignment
                traj_label = Text(regime_name, font=BODY_FONT, font_size=9, color=color_mix, weight=BOLD)
                traj_label.move_to(label_pos).set_opacity(0.0)
                self.play(traj_label.animate.set_opacity(0.85), run_time=0.4)
                regime_labels_placed.add(regime_name)

            self.remove(rocket, active_path)
            faded = make_faded_path(scene_points, color_mix, dashed=(factor >= 1.0))
            self.add(faded)
            self.wait(0.2)

        self.wait(2.0)

        closing_bg = Rectangle(width=config.frame_width, height=config.frame_height).set_fill(PARCHMENT, opacity=1.0).set_stroke(width=0)
        closing_title = Text("Made by Mugambi Ndwiga", font=TITLE_FONT, font_size=32, color=INK)
        closing_handle = Text("@craftsandengineering", font=BODY_FONT, font_size=14, color=INK_FAINT)
        closing_stack = VGroup(closing_title, closing_handle).arrange(DOWN, buff=0.12).move_to(ORIGIN)
        closing_border = RoundedRectangle(width=5.5, height=1.6, corner_radius=0.10).set_fill(color=PARCHMENT_LIGHT, opacity=0.85).set_stroke(color=INK, width=1.2).move_to(ORIGIN)

        self.play(
            FadeOut(VGroup(title_block, threshold_note, current_stats, formula, formula_card, stats_card, earth, earth_shadow, launch_site, launch_rim)),
            FadeIn(closing_bg),
            run_time=0.7
        )
        self.play(FadeIn(closing_border), FadeIn(closing_stack), run_time=0.6)
        self.wait(2.0)

Manim Community v0.20.1

[06/15/26 05:20:27] INFO     Animation 0 : Using cached data (hash :                           ]8;id=925532;file:///usr/local/lib/python3.12/dist-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=383549;file:///usr/local/lib/python3.12/dist-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             4122889543_3466028809_3448355625)                                                     

[06/15/26 05:20:28] INFO     Animation 1 : Using cached data (hash :                           ]8;id=665516;file:///usr/local/lib/python3.12/dist-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=604462;file:///usr/local/lib/python3.12/dist-packages/manim/renderer/cairo_renderer.py#94\94]8;;\
                             441321544_1954709625_2035012139)                                                      

[06/15/26 05:20:30] INFO     Animation 2 : Partial movie file written in                   ]8;id=673088;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=993168;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#601\601]8;;\
                             '/content/media/videos/content/960p24/partial_movie_files/Esc                         
                             apeVelocityStory/441321544_133796724_3789791991.mp4'                                  

[06/15/26 05:20:35] INFO     Animation 3 : Partial movie file written in                   ]8;id=296325;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=331622;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#601\601]8;;\
                             '/content/media/videos/content/960p24/partial_movie_files/Esc                         
                             apeVelocityStory/441321544_2864538691_2907618316.mp4'                                 

...

[06/15/26 05:29:42] INFO     Animation 34 : Partial movie file written in                  ]8;id=588129;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=167916;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#601\601]8;;\
                             '/content/media/videos/content/960p24/partial_movie_files/Esc                         
                             apeVelocityStory/441321544_3151800671_2576986225.mp4'                                 

[06/15/26 05:29:44] INFO     Animation 35 : Partial movie file written in                  ]8;id=972721;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=589661;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#601\601]8;;\
                             '/content/media/videos/content/960p24/partial_movie_files/Esc                         
                             apeVelocityStory/441321544_3877907558_2797895102.mp4'                                 

                    INFO     Combining to Movie file.                                      ]8;id=498547;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=477741;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#753\753]8;;\

[06/15/26 05:29:45] INFO                                                                   ]8;id=835073;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=700829;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#904\904]8;;\
                             File ready at                                                                         
                             '/content/media/videos/content/960p24/EscapeVelocityStory.mp4                         
                             '                                                                                     
                                                                                                                   

                    INFO     Rendered EscapeVelocityStory                                              ]8;id=549242;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene.py\scene.py]8;;\:]8;id=382013;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene.py#278\278]8;;\
                             Played 36 animations                                                                  